In [ ]:
import re
from bisect import bisect_right
import uuid
from collections import defaultdict
from datetime import datetime
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType, IntegerType, LongType, StringType, StructField, StructType, TimestampType
)
from pyspark.sql.window import Window

TIME_PARSER_POLICY = globals().get("TIME_PARSER_POLICY", "CORRECTED")

RUN_ID = str(uuid.uuid4())
STARTED_AT = datetime.utcnow()
spark.conf.set("spark.sql.legacy.timeParserPolicy", TIME_PARSER_POLICY)


def qident(value):
    return "`" + str(value).replace("`", "``") + "`"


def normalise(value):
    return re.sub(r"[^a-z0-9]", "", (value or "").lower())


def append_rows(table_name, rows, schema):
    if rows:
        spark.createDataFrame(rows, schema).write.format("delta").mode("append").saveAsTable(table_name)
